ECE 5464 SP25– HW 5
Due Tuesday, April 29, 2025 – 11:59 PM via Canvas
In the Datasets section of Canvas, you will find a file called “mushroom.csv”. It contains information on
individual samples of a number of mushrooms of a number of different species.
Using the techniques of unsupervised machine learning as we have discussed in class, determine how
many species are in this dataset. Justify your conclusions.
Your submission should a SINGLE Word or pdf file, and it should contain:
1. The answer – how many different species are represented here?
2. A written description of your method.
3. Some analysis that justifies your answer.
4. The source code for any program that you wrote to generate this analysis, using the techniques
of unsupervised learning.
5. Any useful charts and graphs needed to support your answer.

In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import AgglomerativeClustering

In [2]:
path = "/home/kobugi/papaya/ece-5464/project_5/data/mushroom.csv"
df = pd.read_csv(path)

In [3]:
missing_values = df.isnull().sum()
print("Missing values per column:\n", missing_values)

Missing values per column:
 poisonous                   0
cap-shape                   0
cap-surface                 0
cap-color                   0
bruises?                    0
odor                        0
gill-attachment             0
gill-spacing                0
gill-size                   0
gill-color                  0
stalk-shape                 0
stalk-root                  0
stalk-surface-above-ring    0
stalk-surface-below-ring    0
stalk-color-above-ring      0
stalk-color-below-ring      0
veil-type                   0
veil-color                  0
ring-number                 0
ring-type                   0
spore-print-color           0
population                  0
habitat                     0
dtype: int64


#### Need to deal with stalk-root "?"
- Could do kNN

In [4]:
question_marks = df.map(lambda x: x == "?")
question_mark_count = question_marks.sum()
print("Number of '?' values per column:\n", question_mark_count)

Number of '?' values per column:
 poisonous                      0
cap-shape                      0
cap-surface                    0
cap-color                      0
bruises?                       0
odor                           0
gill-attachment                0
gill-spacing                   0
gill-size                      0
gill-color                     0
stalk-shape                    0
stalk-root                  2480
stalk-surface-above-ring       0
stalk-surface-below-ring       0
stalk-color-above-ring         0
stalk-color-below-ring         0
veil-type                      0
veil-color                     0
ring-number                    0
ring-type                      0
spore-print-color              0
population                     0
habitat                        0
dtype: int64


In [5]:
question_marks = df.map(lambda x: x == 0)
question_mark_count = question_marks.sum()
print("Number of '?' values per column:\n", question_mark_count)

Number of '?' values per column:
 poisonous                   0
cap-shape                   0
cap-surface                 0
cap-color                   0
bruises?                    0
odor                        0
gill-attachment             0
gill-spacing                0
gill-size                   0
gill-color                  0
stalk-shape                 0
stalk-root                  0
stalk-surface-above-ring    0
stalk-surface-below-ring    0
stalk-color-above-ring      0
stalk-color-below-ring      0
veil-type                   0
veil-color                  0
ring-number                 0
ring-type                   0
spore-print-color           0
population                  0
habitat                     0
dtype: int64


In [6]:
df.head()

,poisonous,cap-shape,cap-surface,cap-color,bruises?,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


In [7]:
for col in df.columns:
    unique_vals = df[col].unique()
    num_unique = df[col].nunique()
    print(f"{col}: {num_unique} unique values -> {unique_vals}")


poisonous: 2 unique values -> ['p' 'e']
cap-shape: 6 unique values -> ['x' 'b' 's' 'f' 'k' 'c']
cap-surface: 4 unique values -> ['s' 'y' 'f' 'g']
cap-color: 10 unique values -> ['n' 'y' 'w' 'g' 'e' 'p' 'b' 'u' 'c' 'r']
bruises?: 2 unique values -> ['t' 'f']
odor: 9 unique values -> ['p' 'a' 'l' 'n' 'f' 'c' 'y' 's' 'm']
gill-attachment: 2 unique values -> ['f' 'a']
gill-spacing: 2 unique values -> ['c' 'w']
gill-size: 2 unique values -> ['n' 'b']
gill-color: 12 unique values -> ['k' 'n' 'g' 'p' 'w' 'h' 'u' 'e' 'b' 'r' 'y' 'o']
stalk-shape: 2 unique values -> ['e' 't']
stalk-root: 5 unique values -> ['e' 'c' 'b' 'r' '?']
stalk-surface-above-ring: 4 unique values -> ['s' 'f' 'k' 'y']
stalk-surface-below-ring: 4 unique values -> ['s' 'f' 'y' 'k']
stalk-color-above-ring: 9 unique values -> ['w' 'g' 'p' 'n' 'b' 'e' 'o' 'c' 'y']
stalk-color-below-ring: 9 unique values -> ['w' 'p' 'g' 'b' 'n' 'e' 'y' 'o' 'c']
veil-type: 1 unique values -> ['p']
veil-color: 4 unique values -> ['w' 'n' 'o' 'y']


In [8]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import OrdinalEncoder
import numpy as np

# Replace '?' with np.nan
df['stalk-root'] = df['stalk-root'].replace('?', np.nan)

# Create a copy and encode 'stalk-root' for imputation
df_temp = df.copy()

# Ordinal encoding (must be 2D for sklearn)
ordinal = OrdinalEncoder()
encoded = ordinal.fit_transform(df_temp[['stalk-root']])

# Replace column with encoded values
df_temp['stalk-root'] = encoded

# KNN imputation (still needs 2D shape)
imputer = KNNImputer(n_neighbors=5)
imputed = imputer.fit_transform(df_temp[['stalk-root']])

# Inverse transform the imputed values
decoded = ordinal.inverse_transform(imputed)

# Assign back to original DataFrame (flatten to 1D)
df['stalk-root'] = decoded.ravel()


from sklearn.preprocessing import OneHotEncoder


featurenames = df.columns.tolist()
print(featurenames)
encoder = OneHotEncoder()
X = encoder.fit_transform(df[featurenames]).toarray()

In [9]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer


# Select all feature columns
featurenames = df.columns.tolist()

# Create OneHotEncoder with handle_unknown to avoid crashes on new/unseen categories
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Pipeline to ensure consistent preprocessing
preprocessor = Pipeline([
    ('encoder', encoder)
])

# Fit + transform
X = preprocessor.fit_transform(df[featurenames])


In [10]:
from sklearn.metrics import silhouette_score

for k in range(2, 25):
    clus = AgglomerativeClustering(n_clusters=k)
    labels = clus.fit_predict(X)
    score = silhouette_score(X, labels)
    print(f"Clusters: {k}, Silhouette Score: {score:.3f}")


Clusters: 2, Silhouette Score: 0.177
Clusters: 3, Silhouette Score: 0.213
Clusters: 4, Silhouette Score: 0.216
Clusters: 5, Silhouette Score: 0.248
Clusters: 6, Silhouette Score: 0.266
Clusters: 7, Silhouette Score: 0.277
Clusters: 8, Silhouette Score: 0.292
Clusters: 9, Silhouette Score: 0.301
Clusters: 10, Silhouette Score: 0.307
Clusters: 11, Silhouette Score: 0.312
Clusters: 12, Silhouette Score: 0.316
Clusters: 13, Silhouette Score: 0.315
Clusters: 14, Silhouette Score: 0.269
Clusters: 15, Silhouette Score: 0.272
Clusters: 16, Silhouette Score: 0.276
Clusters: 17, Silhouette Score: 0.278
Clusters: 18, Silhouette Score: 0.224
Clusters: 19, Silhouette Score: 0.228
Clusters: 20, Silhouette Score: 0.224
Clusters: 21, Silhouette Score: 0.174
Clusters: 22, Silhouette Score: 0.177
Clusters: 23, Silhouette Score: 0.179
Clusters: 24, Silhouette Score: 0.179


# code from Lecture 24 p 25 
# reference from skleanr.cluster.AgglomerativeClustering
#pathName = 'C:/Data/Mobile/'
#fileName = 'MobilePhoneUsage.xlsx'
#df = pd.read_excel(pathName + fileName, sheet_name='raw data')

from sklearn import cluster

featurenames = df.columns.tolist()
X = df[featurenames].to_numpy()
clus = cluster.AgglomerativeClustering(
compute_distances=True)
ans = clus.fit_predict(X)
df['labels'] = ans
#df.to_excel(path.replace(path+'.xlsx’,'labeled.xlsx'))
plt.title("mushroom dataset analysis")
# plot the top three levels of the dendrogram
plot_dendrogram(clus, truncate_mode="level", p=4)
plt.xlabel("Number of points in node (or index of point if " +
"no parenthesis).")
plt.show()